# 4 — Autoencoder as a pre-processing defence

The last defence leaves the classifier alone and cleans the input instead: a
convolutional autoencoder sits in front of the network, and the classifier
only ever sees a reconstruction.

Two versions of the same architecture, differing only in what they are trained
to remove:

1. **Adversarial** — input is a PGD image, target is the clean original.
2. **Gaussian** — input is the clean image plus `N(0, 0.1)` noise, target is
   the clean original. A denoiser that has never seen an attack.

The comparison asks whether a defence needs to know about the specific
perturbation it is defending against.

Two things to be careful about here, both of which decide whether the numbers
mean anything:

- The autoencoder's decoder ends in a `Sigmoid`, so it outputs `[0, 1]`. Its
  target must also be in `[0, 1]`. This is the whole reason `advlab` keeps
  normalisation inside the model rather than in the dataset transform.
- The attack must be run against the **full defended pipeline**
  (autoencoder + classifier), not against the bare classifier. Attacking the
  classifier alone and then feeding the result through the autoencoder
  measures how well the defence removes a perturbation optimised for a
  different model, which is a much easier test and the usual way an
  input-transformation defence comes to look better than it is.

`advlab` defines every attack on images in `[0, 1]`, so `eps=8/255` is eight
levels of an 8-bit pixel — the standard CIFAR-10 L-infinity budget — and it
means the same thing in all four notebooks.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import advlab
from advlab import CLASSES, DEVICE, accuracy, apgd, cifar10, fgsm, mifgsm, pgd

advlab.set_seed()
print("device:", DEVICE)

In [ ]:
EPS = 8 / 255
BATCH = 128

test_loader = cifar10(train=False, batch_size=256)
train_loader = cifar10(train=True, batch_size=BATCH, fraction=0.5)
classifier = advlab.resnet20(pretrained=True)
print(f"undefended clean accuracy: {accuracy(classifier, test_loader):.2f}%")

In [ ]:
class ConvAutoencoder(nn.Module):
    """32x32 -> 8x8 -> 32x32. Sigmoid output, so it lives in [0, 1]."""

    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),     # 16x16
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),    # 8x8
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


class Defended(nn.Module):
    """Autoencoder then classifier, as one differentiable model.

    Gradients flow through the autoencoder, so an attack on this module is an
    attack on the defence as deployed.
    """

    def __init__(self, autoencoder, classifier):
        super().__init__()
        self.autoencoder = autoencoder
        self.classifier = classifier

    def forward(self, x):
        return self.classifier(self.autoencoder(x))

    @property
    def head(self):
        return self.classifier.head

## Training the two autoencoders

In [ ]:
EPOCHS_AE = 15


def train_autoencoder(corrupt, label):
    """Fit an autoencoder to map `corrupt(images)` back to `images`."""
    advlab.set_seed()
    ae = ConvAutoencoder().to(DEVICE)
    opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
    mse = nn.MSELoss()

    history = []
    for epoch in range(EPOCHS_AE):
        running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            corrupted = corrupt(images, labels)
            ae.train()
            opt.zero_grad()
            loss = mse(ae(corrupted), images)     # target in [0, 1]
            loss.backward()
            opt.step()
            running += loss.item()
        history.append(running / len(train_loader))
        print(f"[{label}] epoch {epoch + 1:2d}/{EPOCHS_AE}"
              f"  MSE {history[-1]:.5f}")
    return ae, history


def pgd_corrupt(images, labels):
    return pgd(classifier, images, labels, eps=EPS, alpha=2 / 255, steps=10)


def gaussian_corrupt(images, labels):
    return (images + 0.1 * torch.randn_like(images)).clamp(0, 1)


ae_adv, hist_adv = train_autoencoder(pgd_corrupt, "adversarial")
ae_gauss, hist_gauss = train_autoencoder(gaussian_corrupt, "gaussian")

In [ ]:
plt.figure(figsize=(6.5, 4))
plt.plot(range(1, EPOCHS_AE + 1), hist_adv, marker="o", ms=3,
         label="trained on PGD images")
plt.plot(range(1, EPOCHS_AE + 1), hist_gauss, marker="s", ms=3,
         label="trained on Gaussian noise")
plt.xlabel("epoch"), plt.ylabel("reconstruction MSE")
plt.title(f"Autoencoder training ({EPOCHS_AE} epochs)")
plt.legend(), plt.grid(alpha=0.3)
plt.show()

## Fine-tuning the head on reconstructions

A reconstruction is not quite a natural image, and the classifier was trained
on natural images. Ten epochs on the final layer, with the body and the
autoencoder frozen, adapt it to what it will actually be fed.

In [ ]:
EPOCHS_FT = 10


def finetune_head(ae, label):
    """Freeze everything but `fc`, fit it on the autoencoder's output."""
    advlab.set_seed()
    clf = advlab.resnet20(pretrained=True)
    defended = Defended(ae, clf).to(DEVICE)

    for param in defended.parameters():
        param.requires_grad = False
    for param in clf.model.fc.parameters():
        param.requires_grad = True

    opt = torch.optim.Adam(clf.model.fc.parameters(), lr=1e-4)
    loss_fn = nn.CrossEntropyLoss()

    history = []
    for epoch in range(EPOCHS_FT):
        running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            ae.eval(), clf.train()
            opt.zero_grad()
            loss = loss_fn(defended(images), labels)
            loss.backward()
            opt.step()
            running += loss.item()
        history.append(running / len(train_loader))
        print(f"[{label}] epoch {epoch + 1:2d}/{EPOCHS_FT}"
              f"  cross-entropy {history[-1]:.4f}")

    for param in defended.parameters():
        param.requires_grad = True
    return defended, history


defended_adv, ft_adv = finetune_head(ae_adv, "adversarial")
defended_gauss, ft_gauss = finetune_head(ae_gauss, "gaussian")

In [ ]:
plt.figure(figsize=(6.5, 4))
plt.plot(range(1, EPOCHS_FT + 1), ft_adv, marker="o", ms=3,
         label="adversarial autoencoder")
plt.plot(range(1, EPOCHS_FT + 1), ft_gauss, marker="s", ms=3,
         label="gaussian autoencoder")
plt.xlabel("epoch"), plt.ylabel("cross-entropy")
plt.title("Classifier head fine-tuning on reconstructions")
plt.legend(), plt.grid(alpha=0.3)
plt.show()

## Evaluation — end-to-end, and against a transferred attack

Both are reported, because the gap between them is the interesting part.

- **End-to-end (white-box):** the attack differentiates through the
  autoencoder. This is the honest number.
- **Transferred:** the attack is built against the bare classifier and then
  passed through the defence. This is what gets reported when the defence is
  treated as a black box in front of the model.

In [ ]:
def report_defended(defended, label):
    a_clean = accuracy(defended, test_loader)
    a_fgsm = accuracy(defended, test_loader, attack=fgsm, eps=EPS)
    a_pgd = accuracy(defended, test_loader, attack=pgd, eps=EPS,
                     alpha=2 / 255, steps=10)
    print(f"{label:28s} clean {a_clean:6.2f}%   FGSM {a_fgsm:6.2f}%"
          f"   PGD {a_pgd:6.2f}%   [end-to-end]")
    return a_clean, a_fgsm, a_pgd


def report_transferred(defended, label):
    """Attack the bare classifier, then evaluate through the defence."""
    defended.eval()
    counts = {"FGSM": 0, "PGD": 0}
    total = 0
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        for name, attack in [("FGSM", fgsm), ("PGD", pgd)]:
            x_adv = attack(classifier, images, labels, eps=EPS)
            with torch.no_grad():
                counts[name] += (defended(x_adv).argmax(1) == labels).sum().item()
        total += labels.numel()
    a_fgsm, a_pgd = (100 * counts["FGSM"] / total, 100 * counts["PGD"] / total)
    print(f"{label:28s} {'':13s}FGSM {a_fgsm:6.2f}%"
          f"   PGD {a_pgd:6.2f}%   [transferred]")
    return a_fgsm, a_pgd


adv_scores = report_defended(defended_adv, "AE trained on PGD")
adv_transfer = report_transferred(defended_adv, "AE trained on PGD")
gauss_scores = report_defended(defended_gauss, "AE trained on Gaussian noise")
gauss_transfer = report_transferred(defended_gauss, "AE trained on Gaussian noise")

In [ ]:
for defended, name in [(defended_adv, "PGD-trained"),
                       (defended_gauss, "Gaussian-trained")]:
    feats, labels, conds = advlab.collect_features(
        defended, test_loader, n_samples=1000, eps=EPS)
    advlab.plot_umap(feats, labels, conds,
                     f"Penultimate features behind the {name} autoencoder")
    plt.show()

## What a reconstruction looks like

In [ ]:
images, labels = next(iter(test_loader))
images, labels = images[:8].to(DEVICE), labels[:8].to(DEVICE)
x_adv = pgd(defended_adv, images, labels, eps=EPS, alpha=2 / 255, steps=10)

with torch.no_grad():
    rows = [
        (images.cpu(), "clean"),
        (ae_adv(images).cpu(), "clean, reconstructed"),
        (x_adv.cpu(), "PGD (end-to-end)"),
        (ae_adv(x_adv).cpu(), "PGD, reconstructed"),
    ]

fig, axes = plt.subplots(4, 8, figsize=(12, 6.5))
for r, (batch, name) in enumerate(rows):
    for c in range(8):
        axes[r, c].imshow(batch[c].permute(1, 2, 0).numpy())
        axes[r, c].set_xticks([]), axes[r, c].set_yticks([])
        if c == 0:
            axes[r, c].set_ylabel(name, fontsize=8)
fig.tight_layout()
plt.show()

## Comparison

In [ ]:
base_clean = accuracy(classifier, test_loader)
base_fgsm = accuracy(classifier, test_loader, attack=fgsm, eps=EPS)
base_pgd = accuracy(classifier, test_loader, attack=pgd, eps=EPS,
                    alpha=2 / 255, steps=10)

print(f"{'model':30s} {'clean':>8s} {'FGSM':>8s} {'PGD':>8s}")
print(f"{'no defence':30s} {base_clean:7.2f}% {base_fgsm:7.2f}% {base_pgd:7.2f}%")
print(f"{'AE (PGD), end-to-end':30s} {adv_scores[0]:7.2f}%"
      f" {adv_scores[1]:7.2f}% {adv_scores[2]:7.2f}%")
print(f"{'AE (PGD), transferred':30s} {'':8s}"
      f" {adv_transfer[0]:7.2f}% {adv_transfer[1]:7.2f}%")
print(f"{'AE (Gaussian), end-to-end':30s} {gauss_scores[0]:7.2f}%"
      f" {gauss_scores[1]:7.2f}% {gauss_scores[2]:7.2f}%")
print(f"{'AE (Gaussian), transferred':30s} {'':8s}"
      f" {gauss_transfer[0]:7.2f}% {gauss_transfer[1]:7.2f}%")

**What the autoencoder can and cannot do.** Its objective is reconstruction
MSE — get the pixels back. Nothing in that objective mentions the classifier's
decision boundary, and an adversarial perturbation is small in exactly the
norm MSE measures. A perturbation of `8/255` contributes on the order of
`1e-3` to the MSE, so an autoencoder can be excellent at its own job while
leaving the perturbation essentially intact. Meanwhile the reconstruction
error on clean images is *larger* than the perturbation it is supposed to
remove, which is why clean accuracy falls: the defence damages the signal more
than the attack does.

**The gap between end-to-end and transferred is the point.** Transferred
numbers look far better, and they are not measuring robustness — they are
measuring that a perturbation optimised for one model is less effective
against a different one. Any input transformation gets this for free. Worse,
the transformation makes gradients harder to estimate without making the model
harder to fool, which is gradient masking (Athalye et al., 2018): the
end-to-end attack still succeeds, it just needs to be pointed at the right
model. If a defence's reported robustness comes from a transferred attack, or
if attacked accuracy ever comes out *above* clean accuracy, the evaluation is
broken rather than the defence being good.

**Whether the corruption matters.** The two autoencoders are trained on
completely different corruptions and end up in much the same place, which says
the limitation is the reconstruction objective rather than the choice of
training noise.